# 26B. Text-Table Baselines — Zero-Shot Lexical, ChouBERT-16, and BGE-M3

**Zero-shot reference points** for the text-table retrieval experiments. No training, no projection heads — just pretrained models and lexical scores evaluated directly on NB 28's GT protocol.

### Models evaluated

| Key | Model | Type | Context | Dim | Notes |
|---|---|---|---|---|---|
| `tfidf_cosine` | TF-IDF + cosine | Lexical (CPU) | full text | sparse | Word-level TF-IDF fit on all texts+tables |
| `bm25` | Okapi BM25 | Lexical (CPU) | full text | n/a | k1=1.5, b=0.75, symmetric two-index scoring |
| `choubert16_meanpool512` | `ChouBERT/ChouBERT-16` | Dense (GPU) | **512 tokens** | 768 | CamemBERT-base, domain-adapted on French plant-health bulletins + tweets; mean-pooled, L2-normalized |
| `bgem3_cls1024` | `BAAI/bge-m3` | Dense (GPU) | **1024 tokens** | 1024 | Raw pretrained BGE-M3, CLS-pooled, L2-normalized — **no heads, no training** |

### Alignment with NB 28

| Element | Match |
|---|---|
| HF cache (`cache_dir=HF_CACHE`) | ✓ `hf_cache2` |
| Logging (single shared log + stdout echo) | ✓ |
| GPU pinning (`cuda:0`) | ✓ |
| GT set (`vars/gt_texttable_records.pkl`, 500 records from NB 28) | ✓ |
| `compute_retrieval_metrics` (identical function) | ✓ |
| Directions (`text -> table`, `table -> text`) | ✓ |
| Per-model `gt_results.csv` + combined `summary_all_models.csv` | ✓ |
| BGE-M3 max-length **1024** | ✓ (NB 28's `TAB_BGE_MAX_LENGTH`) |

### ⚠️ Truncation caveat for ChouBERT-16

CamemBERT's maximum context is **512 tokens**. BSV tables average **800–1400 tokens** after serialization, so ChouBERT-16 sees only the first ~40% of each table. This is the standard zero-shot limitation for a BERT-family model on long-table retrieval — the number is honest but pessimistic relative to what the model could do with chunking. No chunking is applied here (this is a **raw zero-shot baseline**, not a method experiment).

### Output layout

```
OUTPUT_DIR = results/runs/section_4/baselines/zeroshot_texttable/
  zeroshot_texttable_baselines_ft.log
  tfidf_cosine/gt_results.csv
  bm25/gt_results.csv
  choubert16_meanpool512/gt_results.csv
  bgem3_cls1024/gt_results.csv
  summary_all_models.csv
```

### Skip-if-done

Each model produces `gt_results.csv` on first run; subsequent runs skip. Delete a per-model directory to re-evaluate.

In [ ]:
# Resolve paths from the repository root so notebooks work regardless of the kernel CWD.
from pathlib import Path
import os

def _find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "manifests" / "corpus_manifest.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the PhytoPairs repository root (expected data/manifests/corpus_manifest.csv)."
    )

PROJECT_ROOT = _find_project_root()
BSV_ROOT = PROJECT_ROOT / "3.Dataset Construction"
FINETUNING_ROOT = PROJECT_ROOT / "4.Finetuning"
CASE_STUDY_ROOT = PROJECT_ROOT / "5.BBCH case study"
RESULTS_RUNS_ROOT = PROJECT_ROOT / "results" / "runs" / "section_4"
DEFAULT_HF_CACHE = os.environ.get(
    "HF_HOME", str(Path.home() / ".cache" / "huggingface")
)


## Section 0 — Installations

Same pin set as NB 28 / 29, plus `scikit-learn` for TF-IDF.

In [ ]:
# !pip install openpyxl
# !pip install pandas --upgrade
# !pip install accelerate
# !pip install transformers==4.57.6
# !pip install --upgrade huggingface_hub
# !pip install scikit-learn

### Both neural models are ungated — no HuggingFace login required.

`ChouBERT/ChouBERT-16` is a CamemBERT-base model — ungated. `BAAI/bge-m3` is also ungated.

## Section 1 — Setup, logging, config

Log file inside `OUTPUT_DIR`. GPU pinned to `cuda:0` (matches NB 28).

In [1]:
import os
import sys
import time
import math
import pickle
import logging
import gc
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, CamembertTokenizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize

import transformers

In [15]:
# !pip install sentencepiece # required by CamemberTokenizer

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.8 MB/s  0:00:00


In [14]:
# !pip install sentence-transformers==5.4.1 #compatibe avec transformers4.57.X

Defaulting to user installation because normal site-packages is not writeable
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 3.8 MB/s  0:00:00
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 2.2.0
    Uninstalling huggingface_hub-2.2.0:━━━━━━━━━ 0/3 [huggingface-hub]
      Successfully uninstalled huggingface_hub-2.2.0 0/3 [huggingface-hub]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/3 [huggingface-hub]  WARNING: The scripts hf, huggingface-cli and tiny-agents are installed in '/home/sjiang/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this 

In [2]:
transformers.logging.set_verbosity_error()

# --- Paths: define BOTH root and baseline dir up front so log can live inside ---
OUTPUT_ROOT = str(RESULTS_RUNS_ROOT)
OUTPUT_DIR  = os.path.join(OUTPUT_ROOT, "baselines", "zeroshot_texttable")
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(OUTPUT_DIR,  exist_ok=True)

LOG_FORMAT = '%(asctime)s : %(levelname)s : %(message)s'
LOG_FILE   = os.path.join(OUTPUT_DIR, "zeroshot_texttable_baselines_ft.log")

logging.basicConfig(filename=LOG_FILE, level=logging.INFO, format=LOG_FORMAT)
logger = logging.getLogger("img_inbatch_ft")
if not any(type(h) is logging.StreamHandler for h in logger.handlers):
    _sh = logging.StreamHandler(sys.stdout)
    _sh.setFormatter(logging.Formatter(LOG_FORMAT))
    logger.addHandler(_sh)

# --- GPU pinning: always cuda:0 (matches NB 28) ---
CUDA = "cuda:0"
device = CUDA if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(0)
logger.info("Device: %s", device)
if device == CUDA:
    logger.info("GPU: %s (physical index %d of %d visible)",
                torch.cuda.get_device_name(torch.cuda.current_device()),
                torch.cuda.current_device(),
                torch.cuda.device_count())
logger.info("LOG_FILE = %s", LOG_FILE)
logger.info("OUTPUT_DIR = %s", OUTPUT_DIR)

2026-10-22 22:43:46,879 : INFO : Device: cuda:0
2026-10-22 22:43:46,881 : INFO : GPU: Quadro RTX 8000 (physical index 0 of 2 visible)
2026-10-22 22:43:46,883 : INFO : LOG_FILE = /projects/agrifutur/sji/cross_encoder_ft/baselines/zeroshot_texttable/zeroshot_texttable_baselines_ft.log
2026-10-22 22:43:46,884 : INFO : OUTPUT_DIR = /projects/agrifutur/sji/cross_encoder_ft/baselines/zeroshot_texttable


### Config

In [15]:
HF_CACHE = DEFAULT_HF_CACHE
VARS_DIR = str(FINETUNING_ROOT / "vars")
os.makedirs(VARS_DIR, exist_ok=True)

# --- Neural models ---
CHOUBERT_MODEL_NAME  = "ChouBERT/ChouBERT-16"
CHOUBERT_MAX_LENGTH  = 512          # CamemBERT max position embeddings
BGE_MODEL_NAME       = "BAAI/bge-m3"
BGE_MAX_LENGTH       = 1024         # aligned with NB 28's TAB_BGE_MAX_LENGTH
SENTENCE_CAMEMBERT_MODEL_NAME = "dangvantuan/sentence-camembert-large"
SENTENCE_CAMEMBERT_MAX_LEN    = 512
# --- Batching ---
NEURAL_TEXT_BATCH_SIZE = 16

# --- Metrics ---
K_VALUES = [1, 3, 5, 10]

# --- Model keys (also per-model subdirectory names) ---
MODEL_KEYS = [
    "tfidf_cosine",
    "bm25",
    "choubert16_meanpool512",
    "bgem3_cls1024",
    "sentence_camembert_large_meanpool512",   
]

logger.info("Config loaded. CHOUBERT_MAX_LENGTH=%d, BGE_MAX_LENGTH=%d",
            CHOUBERT_MAX_LENGTH, BGE_MAX_LENGTH)
logger.info("MODEL_KEYS=%s", MODEL_KEYS)

2026-10-22 23:09:05,261 : INFO : Config loaded. CHOUBERT_MAX_LENGTH=512, BGE_MAX_LENGTH=1024
2026-10-22 23:09:05,263 : INFO : MODEL_KEYS=['tfidf_cosine', 'bm25', 'choubert16_meanpool512', 'bgem3_cls1024', 'sentence_camembert_large_meanpool512']


## Section 2 — Load GT set (text-table)

Reuses `vars/gt_texttable_records.pkl` from NB 28 — same 500-record set NB 28 evaluates on, so zero-shot numbers are directly comparable to NB 28's trained text-table numbers.

In [4]:
def reload_vars(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)

GT_RECORDS_PATH = os.path.join(VARS_DIR, "gt_texttable_records.pkl")
if not os.path.exists(GT_RECORDS_PATH):
    raise FileNotFoundError(
        f"Missing {GT_RECORDS_PATH}.\n"
        "Run NB 28's Section 6 once to build it, then re-run this cell."
    )

gt_texttable_records = reload_vars(GT_RECORDS_PATH)
logger.info("Loaded %d GT texttable records", len(gt_texttable_records))
assert len(gt_texttable_records) > 0

_s = gt_texttable_records[0]
for _k in ("text", "table_text"):
    assert _k in _s, f"missing key {_k!r} in gt_texttable_records"
logger.info("GT schema OK")

2026-10-22 22:43:55,512 : INFO : Loaded 500 GT texttable records
2026-10-22 22:43:55,513 : INFO : GT schema OK


## Section 3 — Metrics + shared helpers

`compute_retrieval_metrics` is **identical** to NB 28's — same ranking rule, same `R@k` / `MRR`.

In [5]:
def compute_retrieval_metrics(sim_matrix, k_values=K_VALUES):
    N = sim_matrix.shape[0]
    true_scores = sim_matrix.diag().unsqueeze(1)
    ranks = (sim_matrix >= true_scores).sum(dim=1)
    results = {}
    for k in k_values:
        results[f"R@{k}"] = (ranks <= k).float().mean().item()
    results["MRR"] = (1.0 / ranks.float()).mean().item()
    results["N"] = N
    return results


def summary_from_sim(sim):
    """sim: (n_text, n_table). Returns the 2-row DataFrame."""
    return pd.DataFrame([
        {"direction": "text -> table", **compute_retrieval_metrics(sim)},
        {"direction": "table -> text", **compute_retrieval_metrics(sim.T)},
    ])[["direction", "N", "R@1", "R@3", "R@5", "R@10", "MRR"]]


logger.info("Metrics + helpers ready.")

2026-10-22 22:43:57,008 : INFO : Metrics + helpers ready.


## Section 4 — TF-IDF + cosine baseline

Fit a `TfidfVectorizer` on all texts + all tables (shared vocabulary), L2-normalize both sides, cosine similarity = sparse dot product. CPU-only, no model.

In [6]:
def run_tfidf_cosine(gt_records):
    texts  = [r["text"] for r in gt_records]
    tables = [r["table_text"] for r in gt_records]

    t0 = time.time()
    vec = TfidfVectorizer()
    vec.fit(texts + tables)
    X_text  = sk_normalize(vec.transform(texts))
    X_table = sk_normalize(vec.transform(tables))
    sim = (X_text @ X_table.T).toarray()
    sim = torch.tensor(sim, dtype=torch.float32)
    logger.info("[TF-IDF] vocab=%d, sim shape=%s, %.2fs",
                len(vec.vocabulary_), tuple(sim.shape), time.time() - t0)

    return summary_from_sim(sim)

logger.info("run_tfidf_cosine() ready.")

2026-10-22 22:43:58,043 : INFO : run_tfidf_cosine() ready.


## Section 5 — Okapi BM25 baseline

Self-contained Okapi BM25 (k1=1.5, b=0.75). Symmetric scoring: build term statistics over `texts + tables`, then score `text_i` against `table_j` and vice versa. CPU-only.

In [17]:
def _tokenize(s):
    return s.lower().split()


def _bm25_score_matrix(queries, docs, k1=1.5, b=0.75):
    """Returns (len(queries), len(docs)) float32 numpy scores."""
    q_tokens = [_tokenize(q) for q in queries]
    d_tokens = [_tokenize(d) for d in docs]
    N = len(d_tokens)
    avgdl = max(sum(len(d) for d in d_tokens) / max(N, 1), 1e-6)

    df = defaultdict(int)
    for d in d_tokens:
        for t in set(d):
            df[t] += 1
    idf = {t: math.log(1.0 + (N - c + 0.5) / (c + 0.5)) for t, c in df.items()}

    dl = [len(d) for d in d_tokens]
    tf = [defaultdict(int) for _ in d_tokens]
    for i, d in enumerate(d_tokens):
        for t in d:
            tf[i][t] += 1

    S = np.zeros((len(queries), N), dtype=np.float32)
    for qi, q in enumerate(q_tokens):
        for t in q:
            if t not in idf:
                continue
            idf_t = idf[t]
            for di in range(N):
                f = tf[di].get(t, 0)
                if f == 0:
                    continue
                denom = f + k1 * (1.0 - b + b * dl[di] / avgdl)
                S[qi, di] += idf_t * f * (k1 + 1.0) / denom
    return S


def run_bm25(gt_records):
    """Okapi BM25 with TWO separate scoring passes for the two directions.

    BM25 is asymmetric: IDF is fit on the doc side, and the length penalty
    (b=0.75) applies only to the doc side. So score(text_i -> table_j) is NOT
    equal to score(table_j -> text_i). A proper bidirectional BM25 requires
    two independent passes with roles swapped.
    """
    texts  = [r["text"] for r in gt_records]
    tables = [r["table_text"] for r in gt_records]

    t0 = time.time()
    sim_t2t_np = _bm25_score_matrix(texts, tables)   # text queries, table docs
    logger.info("[BM25] forward pass (text -> table) done, %.2fs", time.time() - t0)

    t0 = time.time()
    sim_x2t_np = _bm25_score_matrix(tables, texts)   # table queries, text docs
    logger.info("[BM25] reverse pass (table -> text) done, %.2fs", time.time() - t0)

    sim_t2t = torch.tensor(sim_t2t_np, dtype=torch.float32)   # (n_text,  n_table)
    sim_x2t = torch.tensor(sim_x2t_np, dtype=torch.float32)   # (n_table, n_text)

    logger.info("[BM25] forward sim=%s, reverse sim=%s",
                tuple(sim_t2t.shape), tuple(sim_x2t.shape))

    summary = pd.DataFrame([
        {"direction": "text -> table", **compute_retrieval_metrics(sim_t2t)},
        {"direction": "table -> text", **compute_retrieval_metrics(sim_x2t)},
    ])[["direction", "N", "R@1", "R@3", "R@5", "R@10", "MRR"]]

    return summary

logger.info("run_bm25() ready.")

2026-10-22 23:10:19,078 : INFO : run_bm25() ready.


## Section 6 — ChouBERT-16 meanpool baseline

`ChouBERT/ChouBERT-16` is a CamemBERT-base model (768-dim, 512-token max) further pretrained on French plant-health bulletins + tweets — domain-adjacent to BSV. Mean-pool over non-padding tokens, then L2-normalize. No fine-tuning.

In [8]:
def _meanpool_encode(texts, tokenizer, model, max_length, batch_size):
    embs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = tokenizer(batch, return_tensors="pt", padding=True,
                               truncation=True, max_length=max_length).to(device)
            out = model(**inputs)
            last = out.last_hidden_state                              # (B, L, D)
            mask = inputs["attention_mask"].unsqueeze(-1).float()     # (B, L, 1)
            summed = (last * mask).sum(dim=1)                         # (B, D)
            counts = mask.sum(dim=1).clamp(min=1e-6)                  # (B, 1)
            mean = summed / counts
            mean = F.normalize(mean, p=2, dim=-1)
            embs.append(mean.cpu())
    return torch.cat(embs, dim=0)


def run_choubert16_meanpool(gt_records):
    tokenizer = CamembertTokenizer.from_pretrained(CHOUBERT_MODEL_NAME, cache_dir=HF_CACHE)
    model = AutoModel.from_pretrained(CHOUBERT_MODEL_NAME, cache_dir=HF_CACHE).to(device)
    model.eval()

    texts  = [r["text"] for r in gt_records]
    tables = [r["table_text"] for r in gt_records]

    t0 = time.time()
    txt_emb = _meanpool_encode(texts,  tokenizer, model, CHOUBERT_MAX_LENGTH, NEURAL_TEXT_BATCH_SIZE)
    tbl_emb = _meanpool_encode(tables, tokenizer, model, CHOUBERT_MAX_LENGTH, NEURAL_TEXT_BATCH_SIZE)
    logger.info("[ChouBERT-16] Encoded %d texts + %d tables in %.1fs (dim=%d)",
                len(txt_emb), len(tbl_emb), time.time() - t0, txt_emb.shape[-1])

    sim = txt_emb @ tbl_emb.T
    summary = summary_from_sim(sim)

    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary

logger.info("run_choubert16_meanpool() ready.")

2026-10-22 22:43:59,880 : INFO : run_choubert16_meanpool() ready.


## Section 6B- French SBERT: Sentence CamemBERT

In [16]:
def run_sentence_camembert_large(gt_records):
    """
    dangvantuan/sentence-camembert-large — a French sentence-transformers model
    (CamemBERT-large fine-tuned on French STS). Mean-pooled and L2-normalized
    internally by sentence-transformers. Truncated at 512 tokens (CamemBERT's
    max context). No fine-tuning.
    """
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(
        SENTENCE_CAMEMBERT_MODEL_NAME,
        cache_folder=HF_CACHE,
        device=device,
    )
    model.max_seq_length = SENTENCE_CAMEMBERT_MAX_LEN

    texts  = [r["text"] for r in gt_records]
    tables = [r["table_text"] for r in gt_records]

    t0 = time.time()
    with torch.no_grad():
        txt_emb = model.encode(
            texts,
            batch_size=NEURAL_TEXT_BATCH_SIZE,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).cpu()
        tbl_emb = model.encode(
            tables,
            batch_size=NEURAL_TEXT_BATCH_SIZE,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).cpu()

    logger.info("[Sentence-CamemBERT-Large] Encoded %d texts + %d tables in %.1fs (dim=%d)",
                len(txt_emb), len(tbl_emb), time.time() - t0, txt_emb.shape[-1])

    sim = txt_emb @ tbl_emb.T
    summary = summary_from_sim(sim)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary

logger.info("run_sentence_camembert_large() ready.")

2026-10-22 23:10:10,910 : INFO : run_sentence_camembert_large() ready.


## Section 7 — BGE-M3 raw CLS baseline

Raw `BAAI/bge-m3` with **no projection heads and no training** — exactly as it ships. CLS-pooled (BGE-M3's standard dense embedding), L2-normalized, max-length 1024 (aligned with NB 28's `TAB_BGE_MAX_LENGTH`).

In [9]:
def _cls_encode(texts, tokenizer, model, max_length, batch_size):
    embs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = tokenizer(batch, return_tensors="pt", padding=True,
                               truncation=True, max_length=max_length).to(device)
            out = model(**inputs)
            cls = out.last_hidden_state[:, 0, :]
            cls = F.normalize(cls, p=2, dim=-1)
            embs.append(cls.cpu())
    return torch.cat(embs, dim=0)


def run_bgem3_raw_cls(gt_records):
    tokenizer = AutoTokenizer.from_pretrained(BGE_MODEL_NAME, cache_dir=HF_CACHE)
    model = AutoModel.from_pretrained(BGE_MODEL_NAME, cache_dir=HF_CACHE).to(device)
    model.eval()

    texts  = [r["text"] for r in gt_records]
    tables = [r["table_text"] for r in gt_records]

    t0 = time.time()
    txt_emb = _cls_encode(texts,  tokenizer, model, BGE_MAX_LENGTH, NEURAL_TEXT_BATCH_SIZE)
    tbl_emb = _cls_encode(tables, tokenizer, model, BGE_MAX_LENGTH, NEURAL_TEXT_BATCH_SIZE)
    logger.info("[BGE-M3 raw] Encoded %d texts + %d tables in %.1fs (dim=%d)",
                len(txt_emb), len(tbl_emb), time.time() - t0, txt_emb.shape[-1])

    sim = txt_emb @ tbl_emb.T
    summary = summary_from_sim(sim)

    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary

logger.info("run_bgem3_raw_cls() ready.")

2026-10-22 22:44:00,726 : INFO : run_bgem3_raw_cls() ready.


## Section 8 — Run all baselines (skip-if-done)

In [18]:
_RUNNERS = {
    "tfidf_cosine":           run_tfidf_cosine,
    "bm25":                   run_bm25,
    "choubert16_meanpool512": run_choubert16_meanpool,
    "sentence_camembert_large_meanpool512":    run_sentence_camembert_large, 
    "bgem3_cls1024":          run_bgem3_raw_cls,
}

for key in MODEL_KEYS:
    model_dir = os.path.join(OUTPUT_DIR, key)
    os.makedirs(model_dir, exist_ok=True)
    gt_path = os.path.join(model_dir, "gt_results.csv")

    if os.path.exists(gt_path):
        logger.info("=== [%s] already completed, skipping ===", key)
        continue

    logger.info("=" * 70)
    logger.info("=== Zero-shot text-table evaluation: %s ===", key)
    logger.info("=" * 70)

    summary = _RUNNERS[key](gt_texttable_records)
    summary.insert(0, "model", key)
    summary.to_csv(gt_path, index=False)
    logger.info("[%s] Results:\n%s", key, summary.to_string(index=False))

logger.info("All zero-shot text-table baselines complete.")

2026-10-22 23:11:06,103 : INFO : === [tfidf_cosine] already completed, skipping ===
2026-10-22 23:11:06,105 : INFO : ======================================================================
2026-10-22 23:11:06,105 : INFO : === Zero-shot text-table evaluation: bm25 ===
2026-10-22 23:11:06,106 : INFO : ======================================================================
2026-10-22 23:11:08,712 : INFO : [BM25] forward pass (text -> table) done, 2.61s
2026-10-22 23:11:14,903 : INFO : [BM25] reverse pass (table -> text) done, 6.19s
2026-10-22 23:11:14,919 : INFO : [BM25] forward sim=(500, 500), reverse sim=(500, 500)
2026-10-22 23:11:15,020 : INFO : [bm25] Results:
model     direction   N   R@1   R@3   R@5  R@10     MRR
 bm25 text -> table 500 0.048 0.066 0.074 0.106 0.07423
 bm25 table -> text 500 0.120 0.220 0.260 0.370 0.19978
2026-10-22 23:11:15,021 : INFO : === [choubert16_meanpool512] already completed, skipping ===
2026-10-22 23:11:15,025 : INFO : === [bgem3_cls1024] already complete

config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/400 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/400 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/809k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

2026-10-22 23:12:12,828 : INFO : [Sentence-CamemBERT-Large] Encoded 500 texts + 500 tables in 31.2s (dim=1024)
2026-10-22 23:12:13,224 : INFO : [sentence_camembert_large_meanpool512] Results:
                               model     direction   N   R@1   R@3   R@5  R@10      MRR
sentence_camembert_large_meanpool512 text -> table 500 0.074 0.142 0.174 0.224 0.130617
sentence_camembert_large_meanpool512 table -> text 500 0.094 0.182 0.238 0.336 0.172596
2026-10-22 23:12:13,229 : INFO : All zero-shot text-table baselines complete.


## Section 9 — Combined summary

In [19]:
rows = []
for key in MODEL_KEYS:
    p = os.path.join(OUTPUT_DIR, key, "gt_results.csv")
    if os.path.exists(p):
        rows.append(pd.read_csv(p))
    else:
        logger.warning("[%s] no gt_results.csv found, skipping", key)

if rows:
    summary_all_models = (pd.concat(rows, ignore_index=True)
                            .sort_values(["model", "direction"])
                            .reset_index(drop=True))
    summary_all_models.to_csv(os.path.join(OUTPUT_DIR, "summary_all_models.csv"), index=False)
    logger.info("=== Combined zero-shot text-table baseline summary ===\n%s",
                summary_all_models.to_string(index=False))
    try:
        display(summary_all_models)
    except NameError:
        print(summary_all_models)
else:
    logger.warning("No gt_results.csv files found; nothing to summarize.")

2026-10-22 23:12:31,669 : INFO : === Combined zero-shot text-table baseline summary ===
                               model     direction   N   R@1   R@3   R@5  R@10      MRR
                       bgem3_cls1024 table -> text 500 0.134 0.274 0.354 0.462 0.240422
                       bgem3_cls1024 text -> table 500 0.136 0.268 0.344 0.420 0.234250
                                bm25 table -> text 500 0.120 0.220 0.260 0.370 0.199780
                                bm25 text -> table 500 0.048 0.066 0.074 0.106 0.074230
              choubert16_meanpool512 table -> text 500 0.026 0.054 0.070 0.102 0.055825
              choubert16_meanpool512 text -> table 500 0.006 0.020 0.032 0.048 0.026524
sentence_camembert_large_meanpool512 table -> text 500 0.094 0.182 0.238 0.336 0.172596
sentence_camembert_large_meanpool512 text -> table 500 0.074 0.142 0.174 0.224 0.130617
                        tfidf_cosine table -> text 500 0.154 0.302 0.360 0.478 0.261226
                        tfidf_co

,model,direction,N,R@1,R@3,R@5,R@10,MRR
0,bgem3_cls1024,table -> text,500,0.134,0.274,0.354,0.462,0.240422
1,bgem3_cls1024,text -> table,500,0.136,0.268,0.344,0.420,0.234250
2,bm25,table -> text,500,0.120,0.220,0.260,0.370,0.199780
3,bm25,text -> table,500,0.048,0.066,0.074,0.106,0.074230
4,choubert16_meanpool512,table -> text,500,0.026,0.054,0.070,0.102,0.055825
5,choubert16_meanpool512,text -> table,500,0.006,0.020,0.032,0.048,0.026524
6,sentence_camembert_large_meanpool512,table -> text,500,0.094,0.182,0.238,0.336,0.172596
7,sentence_camembert_large_meanpool512,text -> table,500,0.074,0.142,0.174,0.224,0.130617
8,tfidf_cosine,table -> text,500,0.154,0.302,0.360,0.478,0.261226
9,tfidf_cosine,text -> table,500,0.140,0.242,0.298,0.360,0.218554
